In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a program that performs <code>R</code> rounds of parallel hashing on an array of 32-bit integers using the provided hash function.
    The hash should be applied <code>R</code> times iteratively (the output of one round becomes the input to the next).
</p>

<h2>Implementation Requirements</h2>
<ul>
    <li>External libraries are not permitted</li>
    <li>The <code>solve</code> function signature must remain unchanged</li>
    <li>The final result must be stored in array <code>output</code></li>
</ul>

<h2>Example 1:</h2>
<pre>Input:  numbers = [123, 456, 789], R = 2
Output: hashes = [1636807824, 1273011621, 2193987222]</pre>

<h2>Example 2:</h2>
<pre>Input:  numbers = [0, 1, 2147483647], R = 3
Output: hashes = [96754810, 3571711400, 2006156166]</pre>

<h2>Constraints</h2>
<ul>
    <li>1 ≤ <code>N</code> ≤ 10,000,000</li>
    <li>1 ≤ <code>R</code> ≤ 100</li>
    <li>0 ≤ <code>input[i]</code> ≤ 2147483647</li>

  <li>Performance is measured with <code>N</code> = 5,000,000</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__device__ unsigned int fnv1a_hash(unsigned int input) {
    const unsigned int FNV_PRIME = 16777619;
    const unsigned int OFFSET_BASIS = 2166136261;

    unsigned int hash = OFFSET_BASIS;

    for (int byte_pos = 0; byte_pos < 4; byte_pos++) {
        unsigned char byte = (input >> (byte_pos * 8)) & 0xFFu;
        hash = (hash ^ byte) * FNV_PRIME;
    }

    return hash;
}

__global__ void fnv1a_hash_kernel(const int* input, unsigned int* output, int N, int R) {}

// input, output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(const int* input, unsigned int* output, int N, int R) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    fnv1a_hash_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, output, N, R);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


def fnv1a_hash_u32_scalar(x: cute.Uint32) -> cute.Uint32:
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261
    hash_val = cute.Uint32(OFFSET_BASIS)
    prime = cute.Uint32(FNV_PRIME)
    mask = cute.Uint32(0xFF)
    for byte_pos in range(4):
        byte = (x >> (byte_pos * 8)) & mask
        hash_val = (hash_val ^ byte) * prime
    return cute.Uint32(hash_val)


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, N: cute.Int32, R: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


def fnv1a_hash(x: jax.Array) -> jax.Array:
    FNV_PRIME = jnp.uint32(16777619)
    OFFSET_BASIS = jnp.uint32(2166136261)
    hash_val = jnp.full_like(x, OFFSET_BASIS, dtype=jnp.uint32)

    MASK_FF = jnp.uint32(0xFF)
    for byte_pos in range(4):
        byte = (x >> jnp.uint32(byte_pos * 8)) & MASK_FF
        hash_val = hash_val ^ byte
        hash_val = hash_val * FNV_PRIME

    return hash_val


# input is a tensor on the GPU
def solve(input: jax.Array, N: int, R: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def fnv1a_hash(input: UInt32) -> UInt32:
    alias FNV_PRIME: UInt32 = 16777619
    alias OFFSET_BASIS: UInt32 = 2166136261

    var hash: UInt32 = OFFSET_BASIS

    for byte_pos in range(4):
        var byte_val: UInt32 = (input >> (byte_pos * 8)) & UInt32(0xFF)
        hash = (hash ^ byte_val) * FNV_PRIME

    return hash


def fnv1a_hash_kernel(
    input: UnsafePointer[Int32, MutExternalOrigin],
    output: UnsafePointer[UInt32, MutExternalOrigin],
    N: Int32,
    R: Int32,
):
    pass


# input, output are device pointers (i.e. pointers to memory on the GPU)
@export
def solve(
    input: UnsafePointer[Int32, MutExternalOrigin],
    output: UnsafePointer[UInt32, MutExternalOrigin],
    N: Int32,
    R: Int32,
) raises:
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var blocksPerGrid = ceildiv(N, threadsPerBlock)

    var _kernel = ctx.compile_function[fnv1a_hash_kernel, fnv1a_hash_kernel]()
    ctx.enqueue_function(
        _kernel, input, output, N, R, grid_dim=blocksPerGrid, block_dim=threadsPerBlock
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


def fnv1a_hash(x: torch.Tensor) -> torch.Tensor:
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261
    x_int = x.to(torch.int64)
    hash_val = torch.full_like(x_int, OFFSET_BASIS, dtype=torch.int64)

    for byte_pos in range(4):
        byte = (x_int >> (byte_pos * 8)) & 0xFF
        hash_val = (hash_val ^ byte) * FNV_PRIME
        hash_val = hash_val & 0xFFFFFFFF

    return hash_val.to(torch.int32)


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, R: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def fnv1a_hash(x):
    FNV_PRIME = 16777619
    OFFSET_BASIS = 2166136261

    hash_val = tl.full(x.shape, OFFSET_BASIS, tl.uint32)

    for byte_pos in range(4):
        byte = (x >> (byte_pos * 8)) & 0xFF
        hash_val = (hash_val ^ byte) * FNV_PRIME

    return hash_val


@triton.jit
def fnv1a_hash_kernel(input, output, n_elements, n_rounds, BLOCK_SIZE: tl.constexpr):
    pass


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, N: int, R: int):
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(N, BLOCK_SIZE),)
    fnv1a_hash_kernel[grid](input, output, N, R, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(name="Rainbow Table", atol=0, rtol=0, num_gpus=1, access_tier="free")

    def fnv1a_hash(self, x: torch.Tensor) -> torch.Tensor:
        FNV_PRIME = 16777619
        OFFSET_BASIS = 2166136261
        x_int = x.to(torch.int64)
        hash_val = torch.full_like(x_int, OFFSET_BASIS, dtype=torch.int64)
        for byte_pos in range(4):
            byte = (x_int >> (byte_pos * 8)) & 0xFF
            hash_val = (hash_val ^ byte) * FNV_PRIME
            hash_val = hash_val & 0xFFFFFFFF
        return hash_val

    def reference_impl(self, input: torch.Tensor, output: torch.Tensor, N: int, R: int):
        assert input.shape == (N,)
        assert output.shape == (N,)
        assert input.dtype == torch.int32
        assert output.dtype == torch.uint32

        current = input

        # Apply hash R times
        for _ in range(R):
            current = self.fnv1a_hash(current)

        # Reinterpret the lower 32 bits as uint32
        output.copy_(current.to(torch.int32).view(torch.uint32))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_int32), "in"),
            "output": (ctypes.POINTER(ctypes.c_uint32), "out"),
            "N": (ctypes.c_int, "in"),
            "R": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        input_tensor = torch.tensor([123, 456, 789], device="cuda", dtype=torch.int32)
        output_tensor = torch.empty(3, device="cuda", dtype=torch.uint32)
        return {
            "input": input_tensor,
            "output": output_tensor,
            "N": 3,
            "R": 2,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.int32

        test_cases = []

        # Force users to handle "0 chunks" logic
        test_cases.append(
            {
                "input": torch.tensor([100], device="cuda", dtype=dtype),
                "output": torch.zeros(1, device="cuda", dtype=torch.uint32),
                "N": 1,
                "R": 1,
            }
        )

        test_cases.append(
            {
                "input": torch.tensor([100, 200], device="cuda", dtype=dtype),
                "output": torch.zeros(2, device="cuda", dtype=torch.uint32),
                "N": 2,
                "R": 1,
            }
        )

        # basic_example
        test_cases.append(
            {
                "input": torch.tensor([123, 456, 789], device="cuda", dtype=dtype),
                "output": torch.zeros(3, device="cuda", dtype=torch.uint32),
                "N": 3,
                "R": 2,
            }
        )

        # zero_and_max
        test_cases.append(
            {
                "input": torch.tensor([0, 1, 2147483647], device="cuda", dtype=dtype),
                "output": torch.zeros(3, device="cuda", dtype=torch.uint32),
                "N": 3,
                "R": 3,
            }
        )

        # single_round
        test_cases.append(
            {
                "input": torch.tensor([1, 2, 3, 4, 5], device="cuda", dtype=dtype),
                "output": torch.zeros(5, device="cuda", dtype=torch.uint32),
                "N": 5,
                "R": 1,
            }
        )

        # many_rounds
        test_cases.append(
            {
                "input": torch.randint(0, 2147483647 + 1, (1024,), device="cuda", dtype=dtype),
                "output": torch.zeros(1024, device="cuda", dtype=torch.uint32),
                "N": 1024,
                "R": 50,
            }
        )

        # large_size
        test_cases.append(
            {
                "input": torch.randint(0, 2147483647 + 1, (10000,), device="cuda", dtype=dtype),
                "output": torch.zeros(10000, device="cuda", dtype=torch.uint32),
                "N": 10000,
                "R": 10,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        N, R = 5000000, 10  # Large array with moderate rounds for performance testing
        return {
            "input": torch.randint(0, 2147483647 + 1, (N,), device="cuda", dtype=torch.int32),
            "output": torch.zeros(N, device="cuda", dtype=torch.uint32),
            "N": N,
            "R": R,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
